In [6]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [7]:
len(documents)

72

In [8]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [9]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [10]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [11]:
from evaluation_utils import llm_structured_retry
import json

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)
    
    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

In [12]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:3]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [13]:
sum(usage.total_tokens for usage in usages) / len(usages)

1462.0

In [14]:
import pandas as pd

df_ground_truth = pd.read_csv('data/ground-truth.csv')
ground_truth = df_ground_truth[['question', 'filename']].to_dict(orient='records')

In [15]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)


In [16]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [17]:
q = ground_truth[0]["question"]
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [18]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

In [19]:
def text_search(query, num_results=5):
    return index.search(query, num_results=num_results)

In [20]:
results = text_search(q, num_results=1)
for r in results:
       print(r['filename'])

01-agentic-rag/lessons/03-rag.md


In [22]:
from embedder import Embedder

embed = Embedder()

embeddings_matrix = embed.encode_batch([chunk['content'] for chunk in chunks])


In [25]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(embeddings_matrix, chunks)

In [ ]:

def vector_search(query, num_results=5):
    q_embedding = embed.encode(query)
    return vindex.search(q_embedding, num_results=num_results)

In [27]:
results = vector_search(q, num_results=1)
for r in results:
       print(r['filename'])

01-agentic-rag/lessons/01-intro.md


In [34]:
def compute_relevance(q, search_function):
    filename = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename))

    return relevance

In [35]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [40]:

relevance_total_text = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [41]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    print(f"Hit count: {cnt}")        
    return cnt / len(relevance)


In [42]:
res = hit_rate(relevance_total_text)
print(res)

Hit count: 273
0.7583333333333333


In [44]:

relevance_total_vector = compute_relevance_total(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [45]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [46]:
print(mrr(relevance_total_vector))

0.5646472663139328


In [47]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [52]:
for k in [1, 50, 100, 200]:
    relevance_total_hybride = compute_relevance_total(ground_truth, lambda query, k=k: hybrid_search(query, k=k))
    score = mrr(relevance_total_hybride)
    print(f"k={k}: MRR={score}")

  0%|          | 0/360 [00:00<?, ?it/s]

k=1: MRR=0.6481944444444449


  0%|          | 0/360 [00:00<?, ?it/s]

k=50: MRR=0.637916666666667


  0%|          | 0/360 [00:00<?, ?it/s]

k=100: MRR=0.637916666666667


  0%|          | 0/360 [00:00<?, ?it/s]

k=200: MRR=0.637916666666667
